Jupyter notebook

Author: Chen Xi Ye Xu

These scripts were written by Chen Xi Ye Xu as part of her Master´s Thesis for UAM - Master of Bioinformatics and Computational Biology. 


Initial steps for metagenomic analysis consist of retrieving/downloading the neccesary files. Most of the data we use will be publicly available in databases accesible through NCBI. These chunks of scripts automate the download process and also prepare the metadata and a special file called "manifest.tsv" that Qiime2 especifically requires in order to properly load the sequences.  

In [2]:
#These scripts were written by Chen Xi Ye Xu as part of her Master´s Thesis 
# for UAM - Master of Bioinformatics and Computational Biology.


#Import all used libraries.
import os
import subprocess
import sys
from datetime import datetime
import pandas as pd

## 1. Sequence download.

This first program is used for automated sequences download from NCBI using sratools. 

The only neccesary thing/file is a .csv with the accesion SRA code and the sample IDS (later used to create the manifest.tsv that Qiime2 requires to import data).

In [3]:
#######################################################################
## AUTOMATED SEQUENCE DOWNLOAD
#######################################################################
#This first script is used for automated sequences download from NCBI using sratools. 

#The only neccesary thing/file is a .csv with the accesion SRA code and the sample IDS 
# (later used to create the manifest.tsv that Qiime2 requires to import data).


#Download all SRA sequences from a list.
SRA_filename = input(f"Please input the name of the file with the SRA metadata (.csv) "
                      "(with both run code and sample IDs!): \n")
print("\n")

if not os.path.exists(SRA_filename):
    print(f"ERROR: The file {SRA_filename} was NOT found! Please try again.\n")
    sys.exit()


#Read and check SRA run and sample-id columns.
run_column = input(f"""Please input the column name of the SRA run codes: \n""")
print("\n")
sample_id_column = input(f"""Please input the column name of the sample IDs: \n""")
print("\n")


try:
    data = pd.read_csv(SRA_filename, sep = ",", header = 0)
    run_list = data[run_column].tolist()
    sample_id_list = data[sample_id_column].tolist()

except Exception as e:
    print(f"ERROR: {e}")
    sys.exit()
    
#Summary of the SRA codes file.
if len(run_list) == 0:
    print(f"No codes were found inside the provided list! Exiting the program.")
    sys.exit()
else:
    print(f"A total of {len(run_list)} SRA accession codes were found!\n"
        "Proceeding to download!"
        )


#We create a folder names "Sequences" and download all sequences inside it for a cleaner
# result. 
output_dir = "Sequences"
os.makedirs(output_dir, exist_ok = True)


logs = []
#We validate each SRA code and download it.
for i, sra in enumerate(run_list, 1):
    print(f"{i}: {sra} is being downloaded...")

    cmd = ["fasterq-dump", sra, "--outdir", output_dir]
    check_SRA_validity = subprocess.run(cmd, capture_output = True, text = True )

    if check_SRA_validity.returncode == 0:
        print(f"{i}: {sra} was successfully downloaded!\n")
        logs.append((i, sra, "SUCCESS"))
    else:
        print(f"ERROR {i}: {sra} failed to download! {check_SRA_validity.returncode}\n")
        logs.append((i, sra, "ERROR"))

timestamp = datetime.now()

#Create a new Logs directory.
os.makedirs("./Logs", exist_ok = True)

#Writing the logs to check for errors. 
with open("./Logs/SRA_download_LOGS.txt", "w") as log:
    log.write(f"# {timestamp}\n\n")
    log.write("Number\tSRA accession\tStatus")
    log.write("\n")
    for number, sra_code, status in logs:
        log.write(f"{number}\t{sra_code}\t{status}\n")

print(f"\n\nFinished downloading all available sequences!")
print(f"Please check the log file for any errors!\n")







A total of 31 SRA accession codes were found!
Proceeding to download!
1: SRR37661223 is being downloaded...
1: SRR37661223 was successfully downloaded!

2: SRR37661228 is being downloaded...
2: SRR37661228 was successfully downloaded!

3: SRR37661229 is being downloaded...
3: SRR37661229 was successfully downloaded!

4: SRR37661232 is being downloaded...
4: SRR37661232 was successfully downloaded!

5: SRR37661233 is being downloaded...
5: SRR37661233 was successfully downloaded!

6: SRR37661236 is being downloaded...
6: SRR37661236 was successfully downloaded!

7: SRR37661237 is being downloaded...
7: SRR37661237 was successfully downloaded!

8: SRR37661238 is being downloaded...
8: SRR37661238 was successfully downloaded!

9: SRR37661239 is being downloaded...
9: SRR37661239 was successfully downloaded!

10: SRR37661240 is being downloaded...
10: SRR37661240 was successfully downloaded!

11: SRR37661241 is being downloaded...
11: SRR37661241 was successfully downloaded!

12: SRR

## 2. Creating a manifest.tsv for Qiime2.

Qiime2 requires this special file in order to locate the sequences. In order to make it clearer to use, I personally create the path to the sequences as $PWD/Sequences which means that you will need to run the Qiime2 scripts in the same directory as a folder will all the downloaded sequences.  

In [4]:
#######################################################################
## CREATE A MANIFEST.TSV FOR QIIME2 TO USE
#######################################################################

#Create a MANIFEST file for Qiime2 for all the downloaded sequences. 
print(f"With the downloaded sequences we will create now a manifest.tsv for Qiime2 " 
       "(necessary for importing data).\n")


#######################################################################
## FOR PAIRED-END SEQUENCES
#######################################################################

#We can adapt parts of the code for other file types. 
print(f"In this case the manifest corresponds to an experiment using paired-end data")

#Note that we use ~PWD so the code is flexible, we just need to maintain the 
# relative strcuture of the folders.
with open("Manifest_Qiime2.tsv", "w") as m_line:  
    m_line.write("sample-id\tforward-absolute-filepath\treverse-absolute-filepath\n")
    for i in range(len(run_list)):
       m_line.write(f"{sample_id_list[i]}\t$PWD/../1_SRA_Download/Sequences/{run_list[i]}_1.fastq\t$PWD/../1_SRA_Download/Sequences/{run_list[i]}_2.fastq\n") 

print(f"\nFinished creating the manifest file (Manifest_Qiime2.tsv) for Qiime2!\n")

With the downloaded sequences we will create now a manifest.tsv for Qiime2 (necessary for importing data).

In this case the manifest corresponds to an experiment using paired-end data

Finished creating the manifest file (Manifest_Qiime2.tsv) for Qiime2!



## 3. Formatting metadata for Qiime2. 

Qiime2 requires the first column of the metadata file to match the sample IDs in the manifest. And the names of the columns are also set to certain values so this chunk of code fixes/adapts the metadata file s that Qiime2 won´t crash unexpectedly. 

In [ ]:
def format_qiime2_metadata():
    """ 
    This function takes a metadata file (CSV or TSV) and formats it for QIIME 2 by 
    renaming the specified sample ID column to 'sample-id', reordering the columns
    and saving it as a tab-separated file.
    
    Input:
        - input_file (str): path to the input metadata file (CSV or TSV).

    Output:
        - metadata_Qiime_adapted.tsv (tab-separated file/ .tsv): formatted for QIIME 2.
    
    """


    #1)Ask for name of the file. 
    input_file = input("Enter your metadata file name (with file extension like .txt or .csv): ").strip()

    
    #2)Load the file (it automatically detects csv or tsv).
    try:
        if input_file.endswith(".csv"):
            df = pd.read_csv(input_file)
        else:
            df = pd.read_csv(input_file, sep = None, engine = "python")
    except Exception as e:
        print(f"ERROR reading file: {e}")
        return


    #3)Display available columns from the table.
    print("\nAvailable columns in your metadata file:")
    for col in df.columns:
        print(f"   - {col}")

        
    #4)Ask which column matches the sample IDs used in the manifest.
    target_col = input("\nEnter the exact name of the column you want to use as your sample-id (for example: id): ").strip()
    
    if target_col not in df.columns:
        print(f"\nERROR: Column {target_col} not found in the file.")
        return


    #5)Rename the chosen column to "sample-id" (QIIME 2 requires it)
    df = df.rename(columns = {target_col: "sample-id"})

    
    #6)Reorder columns so "sample-id" is strictly the very first column.
    cols = ["sample-id"] + [col for col in df.columns if col != "sample-id"]
    df = df[cols]

    
    #7)Save as a tab-separated file (.tsv) as required by QIIME 2.
    output_file = "metadata_Qiime_adapted.tsv"
    df.to_csv(output_file, sep = "\t", index = False)

    
    print(f"\nSuccess! Formatted metadata saved as: {output_file}.")
    print("You can now enter metadata_Qiime_adapted.tsv when running Qiime2 bash scripts.")


#Only run code if it is the main script being executed (not imported as a module).
if __name__ == "__main__":
    format_qiime2_metadata()


Available columns in your metadata file[cite: 1]:
   - Run
   - AGE
   - Assay Type
   - AvgSpotLen
   - Bases
   - BIOMATERIAL_PROVIDER
   - BioProject
   - BioSample
   - BioSampleModel
   - Bytes
   - Center Name
   - Collection_Date
   - Consent
   - DATASTORE filetype
   - DATASTORE provider
   - DATASTORE region
   - Experiment
   - geo_loc_name_country
   - geo_loc_name_country_continent
   - geo_loc_name
   - Host
   - Instrument
   - isolation_source
   - lat_lon
   - Library Name
   - LibraryLayout
   - LibrarySelection
   - LibrarySource
   - Organism
   - Platform
   - ReleaseDate
   - create_date
   - version
   - Sample Name
   - sample_type
   - sex
   - SRA Study
   - tissue

Success! Formatted metadata saved as: metadata_Qiime_adapted.tsv.
You can now enter metadata_Qiime_adapted.tsv when running Qiime2 bash scripts.
